![Types of Attention](images/attention.png)

Self-attention in transformers is a technique designed to enhance input representations by enabling each position in a sequence to engage with and determine the relevance of every other position within the same sequence

![Self attention](images/self_attention_1.jpg)

![Self attention](images/self_attention_2.jpg)

To summarize `Self Attention` - we want to go from the simple `Input Embeddings` to a `Context Vector` for each token.

For a given token, we want to identify how much it is influenced by all the other tokens.

Hence, we will represent each token by a context vector, which is a weightd sum of all the other input elements

![Self attention example](images/self_attention_3.png)

![The process of computing attention vectors](images/self_attention_6.png)

## 1. Simplified Self Attention

The primary objective of this section is to demonstrate how the context vector 
Z(2) is calculated using the second input sequence, X(2), as a query

### 1.1 Self Attention for 1 sample input token

In [7]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [10]:
# Step 1: As an example, compute the attention scores for one 
# of the inputs with all the other inputs

query = inputs[1]
attention_scores = torch.empty(inputs.shape[0])

for i, input in enumerate(inputs):
    attention_scores[i] = torch.dot(query, input)

print(attention_scores)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [14]:
# Step 2: Now, normalize the attention scores to get attention weights

# 1. Normalize by dividing by the sum of all weights
attention_weights_1 = attention_scores / attention_scores.sum()
print(attention_weights_1)
print(attention_weights_1.sum())

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
tensor(1.0000)


In [15]:
# However, in production, Softmax is preferred since it is better 
# at handling extreme values and has more desirable gradient properties 
# during training

attention_weights_2 = torch.softmax(attention_scores, dim=0)
print(attention_weights_2)
print(attention_weights_2.sum())

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
tensor(1.)


In [18]:
# Step 3: Compute the `context vector` for the selected input token
# by doing a weighted sum of all the other input vectors

context_vector_1 = torch.empty(query.shape)

for i, input in enumerate(inputs):
    context_vector_1 += input * attention_weights_2[i]

print(context_vector_1)

tensor([0.4419, 0.6515, 0.5683])


![computing the attention weight](images/self_attention_4.png)

## 1.2 Self-Attention for all Input Tokens

![Self Attention for all Input Tokens](images/self_attention_5.png)

![The process of computing attention vectors](images/self_attention_6.png)

In [20]:
# 1. Compute the attention scores

attention_scores = torch.empty(inputs.shape[0], inputs.shape[0])

for i, query in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_scores[i][j] = torch.dot(query, x_j)

print(attention_scores)
# the (i,j) th element is the dot product of token i with token j

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [21]:
# 2. Compute the attention weights

attention_weights = torch.empty(inputs.shape[0], inputs.shape[0])

for idx, attention_score in enumerate(attention_scores):
    attention_weights[idx] = torch.softmax(attention_score, dim=0)

print(attention_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [27]:
# 3. Compute the context vectors

# attn_weights = 6x6 matrix
# input = 6x3 matrix
# context_vector = 6x3 matrix (expected)
# So, we can just do attention_weights x input

# tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
#         [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
#         [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
#         [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
#         [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
#         [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

# inputs = torch.tensor(
#   [[0.43, 0.15, 0.89], # Your     (x^1)
#    [0.55, 0.87, 0.66], # journey  (x^2)
#    [0.57, 0.85, 0.64], # starts   (x^3)
#    [0.22, 0.58, 0.33], # with     (x^4)
#    [0.77, 0.25, 0.10], # one      (x^5)
#    [0.05, 0.80, 0.55]] # step     (x^6)
# )

context_vectors = attention_weights @ inputs
print(context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
